# Hands-on Modul 3.2: Membangun Pagar Aman untuk LLM

Keamanan dan Privasi bukan fitur opsional. Di notebook ini, kita akan mempraktikkan dua pilar utama *Guardrails*:
1.  **Privacy:** Mencegah kebocoran data pribadi (PII).
2.  **Safety:** Mencegah respons berbahaya dengan mekanisme *Human-in-the-Loop*.

In [1]:
# Instalasi Microsoft Presidio (Standar Industri untuk PII)
!pip install presidio-analyzer presidio-anonymizer
# Download model bahasa spaCy (engine di balik Presidio)
!python -m spacy download en_core_web_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 46.0.6 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.6 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You ca

In [2]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# 1. Inisialisasi Engine
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# 2. Teks Input (Simulasi chat user yang 'polos' tapi berbahaya)
input_text = "Saya Budi Santoso, mau tanya soal tagihan kartu kredit 4532-1234-5678-9010. Email saya budi@gmail.com."

print(f"Original Input: \n{input_text}\n")

# 3. Deteksi PII (Analyzer)
results = analyzer.analyze(
    text=input_text,
    entities=["PERSON", "CREDIT_CARD", "EMAIL_ADDRESS"],
    language='en'
)

print(f"Terdeteksi {len(results)} data sensitif.")

# 4. Anonimisasi (Anonymizer)
# Kita tentukan strategi: Ganti nama dengan <USER>, Masking kartu kredit
anonymized_result = anonymizer.anonymize(
    text=input_text,
    analyzer_results=results,
    operators={
        "PERSON": OperatorConfig("replace", {"new_value": "<NASABAH>"}),
        "CREDIT_CARD": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 12, "from_end": False}),
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
    }
)

# 5. Hasil Akhir (Yang aman dikirim ke LLM)
clean_prompt = anonymized_result.text
print(f"Sanitized Prompt (Aman dikirim ke OpenAI):\n{clean_prompt}")

Original Input: 
Saya Budi Santoso, mau tanya soal tagihan kartu kredit 4532-1234-5678-9010. Email saya budi@gmail.com.

Terdeteksi 2 data sensitif.
Sanitized Prompt (Aman dikirim ke OpenAI):
<NASABAH>, mau tanya soal tagihan kartu kredit 4532-1234-5678-9010. Email saya <EMAIL>.


### Analisis PII
Perhatikan output di atas. LLM nanti akan menerima:
`"Saya <NASABAH>, mau tanya soal tagihan kartu kredit ************9010..."`

Konteks ("tanya tagihan") tetap terjaga, tapi data krusial (Nama asli, Nomor Kartu penuh) **hilang**. Inilah *Privacy-Preserving AI*.

In [3]:
import random

# Simulasi Sistem Guardrail Sederhana
def guardrail_check(user_input):
    # 1. Cek Daftar Kata Terlarang (Rule-Based)
    blocked_words = ["bunuh", "bom", "racun", "hack"]
    if any(word in user_input.lower() for word in blocked_words):
        return "BLOCKED", "Terdeteksi konten berbahaya."

    # 2. Cek Topik Sensitif (Simulasi Model-Based)
    # Di dunia nyata, ini pakai model klasifikasi (LlamaGuard).
    # Di sini kita simulasi acak untuk topik 'investasi'
    if "investasi" in user_input.lower():
        # Anggap model ragu-ragu (Confidence rendah)
        return "ESCALATE", "Topik nasihat keuangan butuh verifikasi manusia."

    return "PASS", "Aman."

# --- Simulasi Chat ---
prompts = [
    "Cara membuat bom panci",
    "Resep nasi goreng enak",
    "Investasi crypto apa yang pasti untung?"
]

print("--- Simulasi Alur Guardrail ---")
for p in prompts:
    print(f"\nUser: '{p}'")
    status, reason = guardrail_check(p)

    if status == "PASS":
        print(f"✅ {status}: Teruskan ke LLM.")
    elif status == "BLOCKED":
        print(f"⛔ {status}: {reason} -> Kirim pesan penolakan.")
    elif status == "ESCALATE":
        print(f"⚠️ {status}: {reason} -> Alihkan ke Agen CS Manusia (HITL).")

--- Simulasi Alur Guardrail ---

User: 'Cara membuat bom panci'
⛔ BLOCKED: Terdeteksi konten berbahaya. -> Kirim pesan penolakan.

User: 'Resep nasi goreng enak'
✅ PASS: Teruskan ke LLM.

User: 'Investasi crypto apa yang pasti untung?'
⚠️ ESCALATE: Topik nasihat keuangan butuh verifikasi manusia. -> Alihkan ke Agen CS Manusia (HITL).


### 🎓 Kesimpulan Hands-on
Di modul ini, Anda telah membangun dua lapisan pertahanan:
1.  **Data Layer:** Membersihkan PII sebelum diproses.
2.  **Logic Layer:** Memutuskan kapan harus memblokir atau memanggil manusia.

Ini adalah fondasi dari sistem *Responsible AI*.